In [21]:
import torch
import torch.nn.functional as F
# Softmax: PyTorch
X = torch.tensor([1, 2, 3, 4], dtype=torch.float32)
X_softmax = F.softmax(X, dim=0)
print(X, X_softmax)

tensor([1., 2., 3., 4.]) tensor([0.0321, 0.0871, 0.2369, 0.6439])


Softmax:
$$
\bar{x}_i = \frac{e^{x_i}}{\sum_{j=1}^N e^{x_j}}
$$

In [22]:
import torch
# Softmax: Manual
X = torch.tensor([1, 2, 3, 4], dtype=torch.float32)
X_softmax = torch.exp(X) / torch.exp(X).sum()
print(X, X_softmax)

tensor([1., 2., 3., 4.]) tensor([0.0321, 0.0871, 0.2369, 0.6439])


Safe softmax:
$$
\begin{align*}
mx &= \max(x_{1:N}) \\
\bar{x}_i &= \frac{e^{x_i-mx}}{\sum_{j=1}^N e^{x_j-mx}}
\end{align*}
$$

In [23]:
import torch
# Softmax: Safe
X = torch.tensor([1, 2, 3, 4], dtype=torch.float32)
Y = X - X.max()
Y_exp = Y.exp()
X_softmax = Y_exp / Y_exp.sum()
print(X, X_softmax)


tensor([1., 2., 3., 4.]) tensor([0.0321, 0.0871, 0.2369, 0.6439])


In [24]:
import torch
# Softmax: Online
X = torch.tensor([1, 2, 3, 4], dtype=torch.float32)
X0 = X[:-1]
print(X, X0)
X0_max = X0.max()
X0_exp = (X0 - X0_max).exp()
X0_exp_sum = X0_exp.sum()
X0_softmax = X0_exp / X0_exp_sum
print(X0, X0_softmax)

X_max = torch.max(X0_max, X[-1])
X1_exp = (X[-1] - X_max).exp()
X_exp_sum = X0_exp_sum * (X0_max - X_max).exp() + X1_exp
X0_exp_new = X0_exp * (X0_max - X_max).exp()
X_exp = torch.cat([X0_exp_new, X1_exp.unsqueeze(0)])
X_softmax = X_exp / X_exp_sum
print(X, X_softmax)

tensor([1., 2., 3., 4.]) tensor([1., 2., 3.])
tensor([1., 2., 3.]) tensor([0.0900, 0.2447, 0.6652])
tensor([1., 2., 3., 4.]) tensor([0.0321, 0.0871, 0.2369, 0.6439])


In [35]:
import torch
# Softmax: Block Online
X = torch.tensor([1, 2, 3, 4], dtype=torch.float32)
Xb = torch.split(X, split_size_or_sections=2, dim=0)
Xb0, Xb1 = Xb
print(X, Xb)
print(Xb0, Xb1)

# block 0:
Xb0_max = Xb0.max()
Xb0_exp = (Xb0 - Xb0_max).exp()
Xb0_exp_sum = Xb0_exp.sum()

# block 1:
Xb1_max = Xb1.max()
Xb1_exp = (Xb1 - Xb1_max).exp()
Xb1_exp_sum = Xb1_exp.sum()

# online block softmax:
Xb_max = torch.max(Xb0_max, Xb1_max)
k0 = (Xb0_max - Xb_max).exp()
k1 = (Xb1_max - Xb_max).exp()
Xb0_exp = Xb0_exp * k0
Xb1_exp = Xb1_exp * k1
Xb0_sum = Xb0_exp_sum * k0
Xb1_sum = Xb1_exp_sum * k1
Xb_exp = torch.cat([Xb0_exp, Xb1_exp])
Xb_exp_sum = Xb0_sum + Xb1_sum
X_softmax = Xb_exp / Xb_exp_sum
print(X, X_softmax)

tensor([1., 2., 3., 4.]) (tensor([1., 2.]), tensor([3., 4.]))
tensor([1., 2.]) tensor([3., 4.])
tensor([1., 2., 3., 4.]) tensor([0.0321, 0.0871, 0.2369, 0.6439])


In [37]:
import torch 
import torch.nn.functional as F
# Softmax: Batch Version
X = torch.tensor([i for i in range(8)], dtype=torch.float32)
X = X.view(2, 4)
X_softmax = F.softmax(X, dim=1)
print(X)
print(X_softmax)

tensor([[0., 1., 2., 3.],
        [4., 5., 6., 7.]])
tensor([[0.0321, 0.0871, 0.2369, 0.6439],
        [0.0321, 0.0871, 0.2369, 0.6439]])


In [41]:
import torch
# Softmax: Batch Online
X = torch.tensor([i for i in range(8)], dtype=torch.float32)
X = X.view(2, 4)
b, d = X.shape
Xb0, Xb1 = X[:, :d//2], X[:, d//2:]
print(X, Xb0, Xb1)

Xb0_max, _ = Xb0.max(dim=1, keepdim=True)
Xb0_exp = (Xb0 - Xb0_max).exp()
Xb0_exp_sum = Xb0_exp.sum(dim=1, keepdim=True)

Xb1_max, _ = Xb1.max(dim=1, keepdim=True)
Xb1_exp = (Xb1 - Xb1_max).exp()
Xb1_exp_sum = Xb1_exp.sum(dim=1, keepdim=True)

Xb_max = torch.maximum(Xb0_max, Xb1_max)
k0 = (Xb0_max - Xb_max).exp()
k1 = (Xb1_max - Xb_max).exp()
Xb0_exp = Xb0_exp * k0
Xb1_exp = Xb1_exp * k1
Xb0_sum = Xb0_exp_sum * k0
Xb1_sum = Xb1_exp_sum * k1
Xb_exp = torch.cat([Xb0_exp, Xb1_exp], dim=1)
Xb_exp_sum = Xb0_sum + Xb1_sum
X_softmax = Xb_exp / Xb_exp_sum

print(X)
print(X_softmax)

tensor([[0., 1., 2., 3.],
        [4., 5., 6., 7.]]) tensor([[0., 1.],
        [4., 5.]]) tensor([[2., 3.],
        [6., 7.]])
tensor([[0., 1., 2., 3.],
        [4., 5., 6., 7.]])
tensor([[0.0321, 0.0871, 0.2369, 0.6439],
        [0.0321, 0.0871, 0.2369, 0.6439]])
